# Clean Backtest Colab Runner

This notebook lets you run the packaged backtests with custom date ranges and conditions.

How to use:

1. Adjust the options in the form cell.
2. Run the cells from top to bottom.
3. Check the output tables and download the zip archive if you want the CSV files.


In [ ]:
REPO_URL = "https://github.com/snowballTQ/clean-backtest.git"
BRANCH = "main"
PACKAGE_SUBDIR = "."


In [ ]:
START_DATE = "1985-10-01" # @param {type:"date"}
END_DATE = "2026-03-09" # @param {type:"date"}
BASE_SERIES = "ndx" # @param ["ndx", "composite_splice"]
LEVERAGES = "2,3" # @param {type:"string"}
INCLUDE_ZERO_COST = False # @param {type:"boolean"}
INCLUDE_COST_ADJUSTED = True # @param {type:"boolean"}
RUN_BUYHOLD = True # @param {type:"boolean"}
RUN_TIMING = True # @param {type:"boolean"}
RUN_STAGED = True # @param {type:"boolean"}
SMA_WINDOW = 200 # @param {type:"integer"}
ENTRY_CONFIRM_DAYS = 3 # @param {type:"integer"}
COMMISSION_RATE = 0.001 # @param {type:"number"}
TAX_RATE = 0.22 # @param {type:"number"}
EXPENSE_RATIO = 0.0095 # @param {type:"number"}
BORROW_SPREAD = 1.0 # @param {type:"number"}
SMALL_EXIT_THRESHOLDS = "1.10,1.25,1.50" # @param {type:"string"}
LARGE_EXIT_START = 2.0 # @param {type:"number"}
OUTPUT_NAME = "custom_analysis" # @param {type:"string"}


In [ ]:
from pathlib import Path
import shutil
import subprocess

repo_dir = Path('/content/backtest_repo')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run([
    'git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, str(repo_dir)
], check=True)

package_dir = repo_dir / PACKAGE_SUBDIR
if not package_dir.exists():
    raise FileNotFoundError(f'Missing package directory: {package_dir}')

print(f'Package directory: {package_dir}')


In [ ]:
import subprocess
from pathlib import Path

package_dir = Path('/content/backtest_repo') / PACKAGE_SUBDIR
cmd = [
    'python', 'run_custom_analysis.py',
    '--start-date', START_DATE,
    '--end-date', END_DATE,
    '--base-series', BASE_SERIES,
    '--leverages', LEVERAGES,
    '--sma-window', str(SMA_WINDOW),
    '--entry-confirm-days', str(ENTRY_CONFIRM_DAYS),
    '--commission-rate', str(COMMISSION_RATE),
    '--tax-rate', str(TAX_RATE),
    '--expense-ratio', str(EXPENSE_RATIO),
    '--borrow-spread', str(BORROW_SPREAD),
    '--small-exit-thresholds', SMALL_EXIT_THRESHOLDS,
    '--large-exit-start', str(LARGE_EXIT_START),
    '--output-name', OUTPUT_NAME,
]

if INCLUDE_ZERO_COST:
    cmd.append('--include-zero-cost')
if not INCLUDE_COST_ADJUSTED:
    cmd.append('--skip-cost-adjusted')
if not RUN_BUYHOLD:
    cmd.append('--skip-buyhold')
if not RUN_TIMING:
    cmd.append('--skip-timing')
if RUN_STAGED:
    cmd.append('--include-staged')

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, cwd=package_dir, check=True)


In [ ]:
import pandas as pd
from pathlib import Path

output_dir = Path('/root/backtest_outputs') / OUTPUT_NAME
metrics = pd.read_csv(output_dir / 'metrics.csv')
display(metrics)

config = pd.read_json(output_dir / 'config.json', typ='series')
display(config)


In [ ]:
!cd /root && zip -r /content/backtest_outputs.zip backtest_outputs

from google.colab import files
files.download('/content/backtest_outputs.zip')
